# DKI — train a (nonlinear) model and compute keystoneness

End-to-end Colab workflow:
1. clone + install
2. load bundled synthetic data **or** upload your own abundance table (with an optional read-depth filter)
3. train the replicator model (nonlinear fitness)
4. **predict on every sample**, check reconstruction, run the removal thought experiment
5. compute classical **structural** keystoneness (+ optional null-model z-score)
6. compute **functional** keystoneness via the GCN trait matrix and reproduce the R hex plots
7. visualise and download results

## Which samples do we predict on?
Keystoneness is a **counterfactual derived from the trained model** (remove a
species → re-integrate → measure the Bray–Curtis shift), *not* a label we score
against unseen data. So we run it on **all** communities — there is no leakage
concern, and this matches the paper.

The held-out validation split has exactly one job: **model selection**. A low
`val_bc` tells us the learned assembly rules generalise; only then do we trust
the keystoneness numbers. We compute `*_true` (and so a predicted-vs-true
check) **only when real post-removal communities exist** (`Ptest.csv`). For your
own uploaded abundance table there is no observed removal experiment, so only
the predicted columns are defined — the notebook detects this automatically.

## Relationship to `Keystoneness_computing.R`
The R script computes **two** keystoneness flavours per (sample, species):
* **structural** `Ks` — BC shift in *composition* × (1−p) — ported as
  `classical_structural_keystoneness`.
* **functional** `Kf` — BC shift in *function* space (`q · GCN`) × (1−p) —
  ported as `functional_keystoneness`.
`keystoneness_table(...)` returns both in the R script's `str_*/func_*` layout.

> **Keystoneness is defined on compositions.** Observed abundances come from the
> normalised `data.p_all` / `data.p_test`, so every `p` is a relative abundance
> in `[0,1]` and keystoneness stays ≥ 0 (raw counts would make `p > 1` and turn
> it negative).

## 1. Setup

In [ ]:
import os
if not os.path.exists('/content/DKI'):
    !git clone https://github.com/metagenAu/DKI.git /content/DKI
%cd /content/DKI
# use the main branch (falls through to whatever is checked out if it is missing)
!git fetch origin main 2>/dev/null && git checkout main 2>/dev/null && git pull origin main 2>/dev/null || true
!pip install -q -r requirements.txt
import sys
if '/content/DKI' not in sys.path:
    sys.path.insert(0, '/content/DKI')

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from dki.train import TrainConfig, train
from dki.infer import predict
from dki.data import load_dataset
from dki.keystoneness import (
    classical_structural_keystoneness,
    functional_keystoneness,
    keystoneness_table,
    null_model_keystoneness,
)

print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 2. Data
`USE_BUNDLED=True` uses the repo's 100-taxa synthetic set (which ships the
removal-experiment ground truth, so you also get the `*_true` columns). Set it
to `False` to upload your own abundance CSV.

The DKI loader expects `Ptrain.csv` as **(n_taxa, n_samples)** with no header /
index. The flags below transpose / strip headers from your upload to match.

**`MIN_READS`** is a read-depth quality filter applied to the **raw counts**
*before* normalisation and the train/val split: samples whose total reads
`< MIN_READS` are dropped, then any taxa left with zero detections are dropped.
Set it to `0` to disable. It expects raw counts — leave it at `0` for data that
is already relative abundance, and for the bundled ground-truth set (filtering
reindexes taxa, which would desync the on-disk `Sample_id`/`Species_id`).

In [ ]:
USE_BUNDLED = False        # set False to upload your own
samples_as_rows = True     # only matters when USE_BUNDLED=False
header_row = True          # set True if your CSV has a header row
index_col  = True          # set True if your CSV has a row-label column
MIN_READS  = 10000         # drop samples with < this many total reads (0 = off)

DATA_DIR = '/content/DKI/data' if USE_BUNDLED else '/content/dki_data'

if not USE_BUNDLED:
    from google.colab import files
    os.makedirs(DATA_DIR, exist_ok=True)
    print('Upload your abundance CSV (and optionally a test CSV).')
    uploaded = files.upload()
    for name in uploaded:
        df = pd.read_csv(name,
                         header=0 if header_row else None,
                         index_col=0 if index_col else None)
        arr = df.to_numpy(dtype=np.float32)
        if samples_as_rows:
            arr = arr.T    # -> (n_taxa, n_samples) for the DKI loader
        is_train = ('train' in name.lower()) or (len(uploaded) == 1)
        dst = os.path.join(DATA_DIR, 'Ptrain.csv' if is_train else 'Ptest.csv')
        np.savetxt(dst, arr, delimiter=',')
        print(f'  wrote {dst}  shape={arr.shape}  (taxa, samples)')

print('Data dir:', DATA_DIR, '| MIN_READS =', MIN_READS)
!ls -la $DATA_DIR

## 3. Train
Notes on the config below:
* `nonlinear=True` — fitness becomes `fc2(SiLU(fc1(y)))`; the legacy two-Linear
  cNODE2 collapses to a single linear map, so this is what actually buys you
  nonlinear species interactions.
* `lr=1e-3` — the high-dimensional replicator field diverges at `1e-2`.
* `batch_size=500` is capped to the number of training samples available, so
  smaller datasets simply use the full batch.
* `min_reads=MIN_READS` runs the read-depth filter inside `load_dataset`, so the
  model trains on the filtered taxa set (the loader prints what it dropped).
* `mode` defaults to `'ode'`; the `deq_*` knobs only apply if you set
  `mode='deq'`.

In [ ]:
cfg = TrainConfig(
    data_dir=DATA_DIR, out_dir='/content/results',
    batch_size=500,           # capped to n_train; good coverage, low-variance gradient
    lr=1e-3,                  # high-dim field diverges at 1e-2
    min_lr=1e-5,
    epochs=400,               # early stop will trim it
    early_stop_patience=50,   # tighter now that val_bc is far less noisy
    grad_clip=1.0,
    val_fraction=0.2, seed=0,
    min_reads=MIN_READS,      # read-depth QC filter on raw Ptrain (0 = off)
    save_predictions=True,
    nonlinear=True,           # SiLU fitness — the actually-nonlinear model
    # deq defaults are fine: mode='ode', deq_step=0.5, deq_max_iter=50, deq_tol=1e-6
)
model, result, data = train(cfg)
print(f'Trained on {data.n_species} taxa, {data.p_all.shape[0]} samples '
      f'(after MIN_READS={MIN_READS}).')
print(f'Best val BC: {result.best_val_loss:.6f} at epoch {result.best_epoch}')
print(f'Mean epoch: {np.mean(result.epoch_seconds):.2f}s  '
      f'total: {np.sum(result.epoch_seconds):.0f}s')

In [ ]:
# Quick training curve — confirm val BC actually came down and plateaued.
plt.figure(figsize=(7, 4))
plt.plot(result.train_loss, label='train loss', alpha=0.7)
plt.plot(result.val_loss, label='val BC', alpha=0.9)
plt.axvline(result.best_epoch, ls='--', c='k', lw=1, label=f'best @ {result.best_epoch}')
plt.xlabel('epoch'); plt.ylabel('loss / BC'); plt.legend(); plt.title('Training'); plt.show()

## 4. Predict on every sample + run the species-removal thought experiment
`qtrn` = predicted equilibrium for **all** communities (the intact reference).
`qtst` = predicted equilibrium for each leave-one-species-out assemblage.

Observed abundances are taken from the normalised, **filter-consistent**
`data.p_all` (and `data.p_test`), so they line up with `qtrn` taxon-for-taxon.
Ground-truth `*_true` columns require both real removal communities **and** no
read-depth filtering (filtering reindexes taxa, desyncing the on-disk ids); in
every other case we generate the leave-one-out assemblages ourselves.

In [ ]:
def predict_loo(model, z_all, batch_size=256):
    """Stream every leave-one-present-species-out prediction in chunks.

    Returns ``(qtst, sample_id, species_id)`` with ``qtst`` a single
    ``(n_pairs, n_species)`` float32 numpy array and the ids 1-indexed.

    Why not materialise the full LOO matrix and call ``predict`` once: the
    naive path peaks at THREE ``(n_pairs, N)`` host allocations — the dense
    ``z_loo`` tensor, ``predict``'s concatenated output, and the ``.numpy()``
    copy — which crashes Colab on realistic datasets even though the previous
    fix already streams batches onto the GPU. Here we enumerate the (sample,
    species) pairs first (just ints), preallocate ``qtst`` once, then build
    each ``(batch_size, N)`` LOO chunk and copy its prediction straight into
    the slice. Peak host RAM is the single output array plus one mini-batch.
    """
    z_cpu = z_all.detach().cpu()
    n_samples, n_species = z_cpu.shape

    sample_id, species_id = [], []
    for s in range(n_samples):
        present = torch.nonzero(z_cpu[s] > 0, as_tuple=False).flatten().tolist()
        if len(present) < 2:
            continue                # single-species sample: removing it leaves no community
        for sp in present:
            sample_id.append(s + 1); species_id.append(sp + 1)

    n_pairs = len(sample_id)
    qtst = np.empty((n_pairs, n_species), dtype=np.float32)
    s_idx = np.asarray(sample_id, dtype=np.int64) - 1
    sp_idx = np.asarray(species_id, dtype=np.int64) - 1

    for start in range(0, n_pairs, batch_size):
        end = min(start + batch_size, n_pairs)
        batch = z_cpu[torch.as_tensor(s_idx[start:end])].clone()           # (B, N) CPU
        rows = torch.arange(end - start)
        batch[rows, torch.as_tensor(sp_idx[start:end])] = 0.0
        batch = batch / batch.sum(dim=1, keepdim=True).clamp_min(1e-12)
        out = predict(model, batch).detach().cpu().numpy()                 # (B, N)
        qtst[start:end] = out
        del batch, out

    return qtst, np.asarray(sample_id, dtype=int), np.asarray(species_id, dtype=int)


# Predicted intact composition for EVERY sample, and observed RELATIVE abundances
# straight from the (filter-consistent, simplex-normalised) loader tensors.
qtrn = predict(model, data.z_all).detach().cpu().numpy()      # (n_samples, N)
ptrn = data.p_all.detach().cpu().numpy().T                    # (N, n_samples), cols sum to 1

_truth_files = ['Ztest.csv', 'Ptest.csv', 'Sample_id.csv', 'Species_id.csv']
_have_files = all(os.path.exists(os.path.join(DATA_DIR, f)) for f in _truth_files)
has_truth = _have_files and (MIN_READS <= 0)
if _have_files and MIN_READS > 0:
    print('Note: MIN_READS reindexes taxa/samples, so the on-disk '
          'Sample_id/Species_id/Ptest (original indexing) are NOT used; '
          'generating leave-one-out assemblages from the filtered data instead.')

if has_truth:
    sample_id  = np.loadtxt(os.path.join(DATA_DIR, 'Sample_id.csv'),  delimiter=',').astype(int)
    species_id = np.loadtxt(os.path.join(DATA_DIR, 'Species_id.csv'), delimiter=',').astype(int)
    qtst = predict(model, data.z_test).detach().cpu().numpy()
    ptst = data.p_test.detach().cpu().numpy().T              # (N, n_pairs), cols sum to 1
    print(f'Ground-truth removals present: {len(sample_id)} (sample, species) pairs '
          '— computing predicted AND true keystoneness.')
else:
    qtst, sample_id, species_id = predict_loo(model, data.z_all)
    ptst = np.zeros((ptrn.shape[0], qtst.shape[0]), dtype=np.float32)  # placeholder; pred ignores it
    print(f'Computing predicted keystoneness only over {len(sample_id)} '
          'leave-one-out assemblages.')

assert ptrn.max() <= 1.0 + 1e-6, 'ptrn not compositional — expected columns summing to 1.'

## 4b. Predicted vs observed relative-abundance profiles
Before trusting keystoneness, check the model actually reconstructs communities.
`qtrn[s]` is the predicted equilibrium for sample `s`; `ptrn[:, s]` is the
observed relative abundance — both compositions on the simplex.

These plots use **all** samples (train + val mixed). For an honest *held-out*
view, rerun the scatter with `pred = predict(model, data.z_val)` against
`obs = data.p_val` — that is exactly the `val_bc` the model was selected on.

In [ ]:
# Per-sample profiles: observed vs predicted for the most abundant taxa.
n_show = 4
topk = 25
sel = np.linspace(0, qtrn.shape[0] - 1, n_show).astype(int)

fig, axes = plt.subplots(n_show, 1, figsize=(11, 2.4 * n_show), squeeze=False)
for ax, s in zip(axes[:, 0], sel):
    obs, pred = ptrn[:, s], qtrn[s]
    order = np.argsort(obs)[::-1][:topk]            # rank taxa by observed abundance
    x = np.arange(len(order))
    ax.bar(x - 0.2, obs[order],  width=0.4, label='observed',  color='#2b8cbe')
    ax.bar(x + 0.2, pred[order], width=0.4, label='predicted', color='#e34a33')
    bc = np.abs(obs - pred).sum() / max(np.abs(obs + pred).sum(), 1e-12)
    ax.set_title(f'sample {s + 1}  (Bray-Curtis = {bc:.3f})', fontsize=9)
    ax.set_xticks(x); ax.set_xticklabels((order + 1).astype(str), rotation=90, fontsize=6)
    ax.set_ylabel('rel. abundance')
axes[0, 0].legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Global predicted-vs-observed scatter + distribution of per-sample reconstruction error.
obs_all, pred_all = ptrn.T.reshape(-1), qtrn.reshape(-1)
m = (obs_all > 0) | (pred_all > 0)                  # drop the joint-zero mass
bc_s = (np.abs(ptrn.T - qtrn).sum(1)
        / np.clip(np.abs(ptrn.T + qtrn).sum(1), 1e-12, None))
r = np.corrcoef(obs_all[m], pred_all[m])[0, 1]

fig, (a0, a1) = plt.subplots(1, 2, figsize=(11, 4.5))
a0.scatter(obs_all[m], pred_all[m], s=4, alpha=0.15)
lim = [0, max(float(obs_all.max()), float(pred_all.max()))]
a0.plot(lim, lim, 'k--', lw=1)
a0.set_xlabel('observed rel. abundance'); a0.set_ylabel('predicted rel. abundance')
a0.set_title(f'per-taxon  (Pearson r = {r:.3f})')
a1.hist(bc_s, bins=30, color='#756bb1')
a1.axvline(bc_s.mean(), color='k', ls='--', lw=1, label=f'mean = {bc_s.mean():.3f}')
a1.set_xlabel('per-sample Bray-Curtis (obs vs pred)'); a1.set_ylabel('# samples')
a1.set_title('reconstruction error'); a1.legend()
plt.tight_layout(); plt.show()

## 5. Structural keystoneness
`Ks = BC(q_intact_renorm, q_removed) · (1 − p)`. The predicted value depends only
on the model's predictions (`qtrn`, `qtst`) and observed abundance `p` — it does
**not** use `ptst`, so the placeholder above is harmless. Values are always ≥ 0;
if you see negatives, an abundance column was not a relative abundance.

In [ ]:
ks = classical_structural_keystoneness(qtrn, qtst, ptrn, ptst, sample_id, species_id)
if not has_truth:
    ks = ks.drop(columns=['k_true'])   # undefined without real removal experiments

os.makedirs('/content/results', exist_ok=True)
ks.to_csv('/content/results/keystoneness.csv', index=False)
print('wrote /content/results/keystoneness.csv  rows =', len(ks))
ks.sort_values('k_pred', ascending=False).head(10)

In [ ]:
# If we have ground truth, sanity-check that predicted keystoneness tracks the real thing.
if has_truth:
    r = np.corrcoef(ks['k_pred'], ks['k_true'])[0, 1]
    plt.figure(figsize=(5, 5))
    plt.scatter(ks['k_true'], ks['k_pred'], s=6, alpha=0.3)
    lim = [0, max(ks['k_true'].max(), ks['k_pred'].max())]
    plt.plot(lim, lim, 'k--', lw=1)
    plt.xlabel('k_true'); plt.ylabel('k_pred')
    plt.title(f'Predicted vs true structural keystoneness  (Pearson r = {r:.3f})')
    plt.show()
else:
    print('No ground truth — skipping k_pred vs k_true check.')

## 6. Functional keystoneness (Kf) + reproduce the R hex/Spearman plots
Projects each composition through the gene-copy-number matrix (`q · GCN`) and
measures the BC shift in **function** space — the second half of the R script.

Set `GCN_PATH` to a trait matrix whose **taxa axis matches your abundance table**
(`(n_species, n_functions)` or its transpose; orientation is auto-detected). Note
that with `MIN_READS > 0` the taxa set is filtered, so the GCN must match the
*filtered* taxa; a mismatch is reported and the cell falls back to structural
only.

In [ ]:
GCN_PATH = os.path.join(DATA_DIR, 'GCN.csv')   # point this at your own trait matrix
gcn = None
if os.path.exists(GCN_PATH):
    gcn = np.loadtxt(GCN_PATH, delimiter=',')
    if data.kept_taxa is not None and gcn.shape[0] == len(data.kept_taxa):
        gcn = gcn[data.kept_taxa, :]            # align GCN rows to the filtered taxa
    print('loaded GCN', gcn.shape, '| n_species =', qtrn.shape[1])

have_func = False
try:
    kt = keystoneness_table(qtrn, qtst, ptrn, ptst, sample_id, species_id, gcn=gcn)
    have_func = 'func_pred' in kt.columns
except ValueError as e:
    print('Functional keystoneness skipped:', e)
    kt = keystoneness_table(qtrn, qtst, ptrn, ptst, sample_id, species_id, gcn=None)

if not has_truth:
    kt = kt.drop(columns=[c for c in ('str_true', 'func_true') if c in kt.columns])
kt.to_csv('/content/results/keystoneness_full.csv', index=False)
print('wrote /content/results/keystoneness_full.csv  cols =', list(kt.columns))
kt.head()

In [ ]:
# Reproduce the R figure g1: geom_hex + Spectral, y=x line, Spearman rho.
def hex_panel(ax, x, y, label):
    rho = pd.Series(x).corr(pd.Series(y), method='spearman')
    hb = ax.hexbin(x, y, gridsize=40, cmap='Spectral_r', mincnt=1)
    lim = [0, max(float(np.max(x)), float(np.max(y)))]
    ax.plot(lim, lim, color='#d01c8b')
    ax.set_xlabel(f'$K_{{{label}}}$ (true)'); ax.set_ylabel(f'$K_{{{label}}}$ (prediction)')
    ax.set_title(f'Spearman rho = {rho:.2f}')
    return hb

if has_truth:
    panels = [('str_true', 'str_pred', 's')]
    if have_func:
        panels.append(('func_true', 'func_pred', 'f'))
    fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 4.5), squeeze=False)
    for ax, (xt, yp, lab) in zip(axes[0], panels):
        hb = hex_panel(ax, kt[xt].to_numpy(), kt[yp].to_numpy(), lab)
        fig.colorbar(hb, ax=ax)
    plt.tight_layout(); plt.show()
else:
    print('No ground truth — the R hex plot compares true vs predicted, so it needs Ptest.')

## 7. Median keystoneness per species (community-specificity view)
The paper ranks taxa by **median keystoneness across communities**. Species
that are consistently high are the candidate keystone taxa.

In [ ]:
med = (ks.groupby('species')['k_pred']
         .median().sort_values(ascending=False))
top = med.head(20)
plt.figure(figsize=(9, 4))
plt.bar(top.index.astype(str), top.values)
plt.xlabel('species (1-indexed, filtered set)'); plt.ylabel('median k_pred')
plt.title('Top 20 species by median structural keystoneness'); plt.xticks(rotation=60)
plt.tight_layout(); plt.show()
top.to_frame('median_k_pred')

## 8. (Optional) Null-model z-score calibration
Answers a sharper question: *is this species more impactful than equally-
abundant species in the same community?* It builds the leave-one-out
assemblages internally from the trained model, so it needs no `Ztest.csv`.
Reported **alongside** the classical score, never replacing it.

In [ ]:
RUN_NULL_MODEL = True   # a bit slower: extra model solves per (sample, species)

if RUN_NULL_MODEL:
    predict_fn = lambda z: predict(model, z)
    ks_z = null_model_keystoneness(
        predict_fn, data.z_all, ptrn=ptrn,
        sample_id=sample_id, species_id=species_id,
        n_null=50, seed=0,
    )
    ks_z.to_csv('/content/results/keystoneness_zscore.csv', index=False)
    print('wrote /content/results/keystoneness_zscore.csv')
    display(ks_z.sort_values('k_zscore', ascending=False).head(10))

## 9. Download results

In [ ]:
!ls -la /content/results
try:
    from google.colab import files
    files.download('/content/results/keystoneness_full.csv')
except Exception as e:
    print('Not in Colab or download skipped:', e)